<a href="https://colab.research.google.com/github/wuriley001/gaming-support-llm-finetuning/blob/main/notebooks/gaming_intent_lora.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 一、检查 GPU

In [1]:
import torch

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch version: 2.11.0+cu130
CUDA available: True
GPU: Tesla T4


# 二、安装项目需要的库

In [2]:
%pip install -q -U transformers datasets peft trl accelerate bitsandbytes scikit-learn pandas matplotlib

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 8.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.3/80.3 kB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 119.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 45.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 835.4/835.4 kB 59.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 925.5/925.5 kB 67.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 19.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 123.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 126.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 130.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires

# 三、检查库是否安装成功

In [3]:
import transformers
import datasets
import peft
import trl
import accelerate
import bitsandbytes
import sklearn
import pandas as pd
import matplotlib

print("transformers:", transformers.__version__)
print("datasets:", datasets.__version__)
print("peft:", peft.__version__)
print("trl:", trl.__version__)
print("accelerate:", accelerate.__version__)

print("\nEnvironment check passed.")

transformers: 5.18.0
datasets: 5.0.1
peft: 0.21.2
trl: 1.14.1
accelerate: 1.15.0

Environment check passed.


# 四、第一次加载 Qwen

In [4]:
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto"
)

model.eval()

print("Model loaded successfully.")
print("Model device:", model.device)

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Model loaded successfully.
Model device: cuda:0


# 五、第一次 inference
```
Gaming support query
        ↓
Tokenizer
        ↓
Qwen2.5-0.5B-Instruct
        ↓
GPU inference
        ↓
Text response
```
现在发生的是：
Qwen 原始模型 → inference

还完全没有：
training fine-tuning LoRA

In [5]:
messages = [
    {
        "role": "system",
        "content": "You are a helpful assistant for gaming peripheral support."
    },
    {
        "role": "user",
        "content": "My wireless mouse keeps disconnecting while I am gaming. Briefly describe the issue."
    }
]

text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

inputs = tokenizer(
    [text],
    return_tensors="pt"
).to(model.device)

with torch.inference_mode():
    generated_ids = model.generate(
        **inputs,
        max_new_tokens=64,
        do_sample=False
    )

new_tokens = generated_ids[:, inputs["input_ids"].shape[1]:]

response = tokenizer.batch_decode(
    new_tokens,
    skip_special_tokens=True
)[0]

print(response)

The issue with your wireless mouse disconnecting during gaming could be due to several reasons:

1. **Network Issues**: The network connection might not be stable or strong enough, causing the mouse to disconnect.

2. **Software Conflicts**: There may be conflicts between the operating system and the software that controls the mouse (e
